# Post-competition re-run — resurrecting ARM T

Unofficial slots. The competition closed 2026-08-16 and our standing (#120/500, private
0.910686008) is locked. Plan and pre-committed reads: `experiments/RERUN_PLAN.md`.

**Set the runtime to GPU** (Runtime → Change runtime type → T4). 25 runs on CPU will take hours.

### The one rule this notebook exists to enforce

`submissions/preds/*.npz` is copied to Drive **after every single run**, not once at the end.
Last time the download cell took only the CSVs, `submissions/preds/` is gitignored, and every
bundle from the ARM T lane died with the VM. That single missing copy is why ARM T's
counterfactual could not be settled offline, and why this notebook exists.

### The recipe, taken from `experiments/reproduce_champion.sh` — not guessed

ARM T ran **on the permanence champion**, not on the bare config default. `run_pipeline.py`
defaults to `--model gbdt` (the superseded baseline) and the committed config reproduces the
original **24**-channel model. The permanence channel is switched on explicitly:

    PERM = --set seq.channels.permanence=true --set seq.channels.cdf_taus=[-21.0]   # -> 25 ch

**Watch the width fingerprint in every log: `seq input width: 25 channels/month`.** If a tcons run
logs 24, `PERM` did not attach and the run is *invalid*, not merely different.

`seq.distill.enable=true` also **hard-requires** `seq.distill.teacher=<preds .npz>` — it exits
otherwise — so the teacher is built first in Stage A.

## Cell 1 — mount Drive, unpack code, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where bundles persist. Survives the VM.
DRIVE = '/content/drive/MyDrive/geoai_rerun'
import os
os.makedirs(DRIVE + '/preds', exist_ok=True)
os.makedirs(DRIVE + '/submissions', exist_ok=True)
print('persisting to', DRIVE)

In [ ]:
# Upload geoai-aquaculture-code.zip + Train.csv + Test.csv + SampleSubmission.csv first
# (sidebar uploader, or drop them in Drive and cp them across).
!mkdir -p /content/geoai && unzip -o -q /content/geoai-aquaculture-code.zip -d /content/geoai
!pip -q install lightgbm==4.6.0 catboost==1.2.10 pyyaml
!mkdir -p /content/geoai/data/raw
!cp /content/Train.csv /content/Test.csv /content/SampleSubmission.csv /content/geoai/data/raw/
%cd /content/geoai
!mkdir -p submissions/preds
!python -c "import torch; print('cuda:', torch.cuda.is_available())"

## Cell 2 — seeding proof

Two consecutive smoke runs must print an **identical** `final_oof`. Do not proceed if they differ.

In [ ]:
!python run_pipeline.py --smoke --name smoke_a 2>&1 | grep -i "final_oof"
!python run_pipeline.py --smoke --name smoke_b 2>&1 | grep -i "final_oof"

## Cell 3 — the run harness

`persist()` runs after **every** job. If the VM dies at run 14, runs 1–13 are already safe, and
re-executing the cell skips whatever is already in Drive.

In [ ]:
import subprocess, shutil, glob, os, time, re

PERM   = ['--set', 'seq.channels.permanence=true', '--set', 'seq.channels.cdf_taus=[-21.0]']
SEEDS5 = [42, 7, 13, 21, 29]
SEEDS10 = [42, 13, 7, 21, 29, 3, 11, 17, 31, 37]

def persist():
    # The step that was missing last time. Called after EVERY run.
    for f in glob.glob('/content/geoai/submissions/preds/*.npz'):
        shutil.copy(f, DRIVE + '/preds/')
    for f in glob.glob('/content/geoai/submissions/*.csv'):
        shutil.copy(f, DRIVE + '/submissions/')

def run(name, extra):
    if os.path.exists(f'{DRIVE}/preds/preds_{name}.npz'):
        print(f'skip {name} (already in Drive)')
        shutil.copy(f'{DRIVE}/preds/preds_{name}.npz', '/content/geoai/submissions/preds/')
        return True
    t0 = time.time()
    cmd = ['python', 'run_pipeline.py', '--full', '--model', 'seq', '--name', name] + extra
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'FAILED {name}'); print(r.stdout[-3000:]); print(r.stderr[-3000:]); return False
    width = re.findall(r'input width: (\d+) channels', r.stdout)
    oof   = re.findall(r'final_oof[^\n]*', r.stdout)
    flag  = '' if width and width[-1] == '25' else '  <-- WIDTH NOT 25, RUN IS INVALID'
    print(f'{name:20s} {time.time()-t0:6.0f}s  width={width[-1] if width else "?"}  '
          f'{oof[-1] if oof else ""}{flag}')
    persist()
    return True

## Cell 4 — Stage A: the teacher (5 runs)

`champion_perm_seedavg5` — scored **public 0.899882** when we submitted it. ARM D distils against
this; ARM T does not need it. One round of distillation only: the teacher is always the
*non-distilled* pool.

In [ ]:
for s in SEEDS5:
    run(f'perm_single_s{s}', PERM + ['--set', f'seed={s}'])

!python tools/seed_average.py --variant perm_single --name champion_perm_seedavg5
persist()
TEACHER = 'submissions/preds/preds_champion_perm_seedavg5.npz'
print('teacher exists:', os.path.exists('/content/geoai/' + TEACHER))

## Cell 5 — Stage B: ARM T at 10 seeds

The `Var_k(logit)` cross-view penalty on the 1030 unlabeled test rows, at the iter41 settings
(`lambda_u=0.5, K_u=2, min_len=4, warmup_frac=0.5`). A reproduction, not a new arm.

Seed 42 must come back at **public 0.914179** when uploaded — that is control S1.

In [ ]:
TCONS = PERM + ['--set', 'seq.transduct.enable=true', '--set', 'seq.transduct.lambda_u=0.5']
for s in SEEDS10:
    if not run(f'tcons_s{s}', TCONS + ['--set', f'seed={s}']):
        break

## Cell 6 — Stage C: ARM D at α=1.5, 10 seeds

`champion_distill_a15_seedavg5` scored **public 0.910837** at 5 seeds — our best legal champion.
Ten seeds tests recommendation 9 (seed-averaging is the lever worth pulling).

In [ ]:
DISTILL = PERM + ['--set', 'seq.distill.enable=true',
                  '--set', f'seq.distill.teacher={TEACHER}',
                  '--set', 'seq.distill.alpha=1.5']
for s in SEEDS10:
    if not run(f'distill15_s{s}', DISTILL + ['--set', f'seed={s}']):
        break

## Cell 7 — pool, guard, and the combiner question

`assert_pool_sane` trips on the ARM T *replay* at +24 rows past the widened member envelope. This
is the first time it meets the real thing.

**`--guard warn` is deliberate and required here.** `calibrated_pool` defaults to `guard="raise"`,
so the default `seed_average.py` would *abort* on exactly the artifact S2 needs — the tcons pool is
the case the guard was built to refuse. For this diagnostic round we want the verdict recorded, not
the run aborted. Keep the raising default everywhere else; this is the one place it is overridden,
and it is overridden out loud.

And the question Agent A could not answer: the combiner null (probability average vs
geometric-mean-of-odds vs pool-then-calibrate landing **0–2 rows of 1030** apart) was measured only
on families *without* the variance penalty. ARM T is the one lane whose whole mechanism is logit
compression. Does the null hold here, or break exactly where the mechanism predicts?

In [ ]:
!python tools/seed_average.py --variant tcons     --name tcons_seedavg10     --guard warn
!python tools/seed_average.py --variant distill15 --name distill15_seedavg10 --guard warn
persist()

In [ ]:
# Did the guard fire, and on which family? This is the load-bearing read of Cell 7.
!python tools/seed_average.py --variant tcons --name _guardcheck --guard warn 2>&1 | grep -Ei "POOLED|pos-rate|defect|escape|guard|WARN" | head -20

In [ ]:
!python tools/repool.py --variant tcons     2>&1 | tail -45

In [ ]:
!python tools/repool.py --variant distill15 2>&1 | tail -25

## Cell 8 — the criterion table, recorded BEFORE any upload

Recommendation 3: select on **OOF AUC alone**. OOF F1@0.5 measured anti-predictive
(ρ = −0.420, P = 0.017) and the offline composite is worse than its own AUC term.

Print what each criterion *would* pick, now, so the leaderboard comparison is honest rather than
reconstructed after the fact.

In [ ]:
import numpy as np, glob, os
from sklearn.metrics import roc_auc_score, f1_score

for fam in ('tcons', 'distill15'):
    rows = []
    for f in sorted(glob.glob(f'/content/geoai/submissions/preds/preds_{fam}_s*.npz')):
        d = np.load(f, allow_pickle=True)
        y, p, pt = d['y'], d['oof_prob'], d['p_test_raw']
        auc = roc_auc_score(y, p)
        f1 = f1_score(y, (p >= 0.5).astype(int))
        rows.append((os.path.basename(f)[6:-4], auc, f1, 0.6 * f1 + 0.4 * auc,
                     float((pt >= 0.5).mean())))
    if not rows:
        print(f'no {fam} bundles'); continue
    print(f'\n=== {fam} ===')
    print(f"{'member':18s} {'OOF AUC':>9s} {'OOF F1':>9s} {'composite':>10s} {'test posrate':>13s}")
    for n, a, f1, c, pr in rows:
        print(f'{n:18s} {a:9.6f} {f1:9.6f} {c:10.6f} {pr:13.4f}')
    pr_all = [r[4] for r in rows]
    print(f'member pos-rate range [{min(pr_all):.4f}, {max(pr_all):.4f}]  '
          f'(true test prevalence 0.5437 -- DIAGNOSTIC comparison only, never a selector)')
    pick = lambda i: max(rows, key=lambda r: r[i])[0]
    print(f'  OOF AUC   (R3, the only legal criterion) -> {pick(1)}')
    print(f'  OOF F1    (retired, anti-predictive)     -> {pick(2)}')
    print(f'  composite (retired)                      -> {pick(3)}')

## Cell 9 — compliance audit, then download

Reads the CSV directly and trusts none of our own code: the binary column must be the literal 0.5
cut of the probability column.

In [ ]:
import pandas as pd
def audit(path):
    df = pd.read_csv(path)
    ok = (df['TargetF1'].astype(int) == (df['TargetRAUC'] >= 0.5).astype(int)).all()
    print(f"{os.path.basename(path):45s} literal-0.5={bool(ok)}  "
          f"pos-rate={df['TargetF1'].mean():.4f}  distinct_p={df['TargetRAUC'].nunique()}/{len(df)}")
    return ok

for f in sorted(glob.glob('/content/geoai/submissions/submission_*.csv')):
    if any(k in f for k in ('tcons', 'distill15', 'perm_seedavg5')):
        audit(f)

In [ ]:
# Upload order is fixed by experiments/RERUN_PLAN.md section 3. S1 FIRST: if the reproduction
# control misses public 0.914179 by more than 0.001, STOP -- the environment drifted and nothing
# else this round is interpretable.
from google.colab import files
for f in ['submission_tcons_s42.csv',             # S1 control       -> expect public 0.914179
          'submission_tcons_seedavg10.csv',       # S2 resurrection
          'submission_<best_oof_auc_member>.csv', # S3 criterion test -- edit from Cell 8
          'submission_distill15_seedavg10.csv']:  # S4 seed-averaging lever
    try:
        files.download('/content/geoai/submissions/' + f)
    except Exception as e:
        print('skip', f, e)

## After the scores land

Append to `experiments/RERUN_PLAN.md` §5: filename, public, private, AUC and F1 sub-columns, and
— the part that matters — **whether the pre-committed read was met**. Deciding what a number meant
after seeing it is the exact failure this round exists to avoid.

Copy `MyDrive/geoai_rerun/preds/` home. That directory is what makes the next post-mortem
possible. **Do not commit it**: `.npz` carries `y` and `test_ids`.